# Kvasir-SEG Dataset Prep: Metadata, Splits, and Manifests

Local-first dataset preparation for Kvasir-SEG.\n\nThis notebook:\n- resolves local extracted dataset or local zip\n- optionally downloads zip if enabled\n- builds deterministic train/val/test splits\n- writes metadata/manifests/split hash artifacts under `0_dataset_prep/out/`

In [1]:
import sys
from pathlib import Path

def _bootstrap_kvasir_seg_path() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / 'utils' / 'segmentation_common.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
        alt = p / 'Prototyping_reformat' / 'DatasetAnalysis' / 'Kvasir_SEG'
        if (alt / 'utils' / 'segmentation_common.py').exists():
            if str(alt) not in sys.path:
                sys.path.insert(0, str(alt))
            return alt
    raise RuntimeError('Could not locate Kvasir_SEG utils path from current working directory.')

BOOTSTRAP_ROOT = _bootstrap_kvasir_seg_path()
print('BOOTSTRAP_ROOT:', BOOTSTRAP_ROOT)

BOOTSTRAP_ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG


In [2]:

import os
import json
from pathlib import Path
import pandas as pd

from utils.segmentation_common import (
    find_kvasir_seg_root,
    ensure_kvasir_seg_data,
    ensure_kvasir_sessile_zip,
    load_bboxes_json,
    build_kvasir_seg_metadata,
    write_metadata_artifacts,
)

ROOT = find_kvasir_seg_root()
SPLIT_SEED = int(os.getenv('SPLIT_SEED', '42'))
TRAIN_RATIO = float(os.getenv('TRAIN_RATIO', '0.8'))
VAL_RATIO = float(os.getenv('VAL_RATIO', '0.1'))
ALLOW_DOWNLOAD = os.getenv('ALLOW_DOWNLOAD', '0') == '1'
MASK_THRESHOLD = int(os.getenv('MASK_THRESHOLD', '127'))

print('ROOT:', ROOT)
print('SPLIT_SEED:', SPLIT_SEED)
print('TRAIN_RATIO:', TRAIN_RATIO, 'VAL_RATIO:', VAL_RATIO)
print('ALLOW_DOWNLOAD:', ALLOW_DOWNLOAD)


ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG
SPLIT_SEED: 42
TRAIN_RATIO: 0.8 VAL_RATIO: 0.1
ALLOW_DOWNLOAD: False


In [3]:

# Resolve Kvasir-SEG folder
kvasir_seg_dir = ensure_kvasir_seg_data(ROOT, allow_download=ALLOW_DOWNLOAD)
print('Using Kvasir-SEG dir:', kvasir_seg_dir)

# Resolve bbox json
bbox_candidates = [
    kvasir_seg_dir / 'kavsir_bboxes.json',
    ROOT / '0_dataset_prep' / 'Kvasir-SEG' / 'kavsir_bboxes.json',
    ROOT / '0_dataset_prep' / 'out' / 'Kvasir-SEG' / 'kavsir_bboxes.json',
]
bbox_path = next((p for p in bbox_candidates if p.exists()), None)
print('BBox path:', bbox_path)

bboxes = load_bboxes_json(bbox_path) if bbox_path else {}
print('Loaded bbox entries:', len(bboxes))


Using Kvasir-SEG dir: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/Kvasir-SEG
BBox path: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/Kvasir-SEG/kavsir_bboxes.json
Loaded bbox entries: 1000


In [4]:

# Build metadata
meta_df = build_kvasir_seg_metadata(
    data_dir=kvasir_seg_dir,
    bboxes=bboxes,
    seed=SPLIT_SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    threshold=MASK_THRESHOLD,
)

artifact_paths = write_metadata_artifacts(ROOT, meta_df)

print('Rows:', len(meta_df))
print(meta_df.groupby('split').size())
print('Artifacts:')
for k, v in artifact_paths.items():
    print('-', k, ':', v)


Rows: 1000
split
test     100
train    800
val      100
dtype: int64
Artifacts:
- raw_csv : /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/metadata/metadata_raw.csv
- enriched_csv : /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/metadata/metadata_enriched.csv
- manifest_csv : /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/manifests/image_mask_manifest.csv
- split_hash_txt : /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/metadata/split_hash.txt
- splits_dir : /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/splits


In [5]:

# Also stage Kvasir-Sessile zip path when available (for later cross-dataset notebook)
sessile_zip = ensure_kvasir_sessile_zip(ROOT, allow_download=False)
print('Kvasir-Sessile zip present:', bool(sessile_zip))
print('Kvasir-Sessile zip path:', sessile_zip)


Kvasir-Sessile zip present: True
Kvasir-Sessile zip path: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/kvasir-sessile.zip


In [6]:

# Preview
display(meta_df.head(5))
print('Unique images:', meta_df['img_id'].nunique())
print('Mean mask area ratio:', float(meta_df['mask_area_ratio'].mean()))


,img_id,image_path,mask_path,width,height,fg_pixels,total_pixels,mask_area_ratio,component_count,bbox_count,split,split_seed
0,cju0qkwl35piu0993l0dewei2,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,622,529,93043,329038,0.282773,1,1,train,42
1,cju0qoxqj9q6s0835b43399p4,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,1348,1070,390295,1442360,0.270595,1,1,test,42
2,cju0qx73cjw570799j4n5cjze,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,619,529,104400,327451,0.318826,1,1,train,42
3,cju0roawvklrq0799vmjorwfv,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,622,528,35632,328416,0.108497,3,3,train,42
4,cju0rx1idathl0835detmsp84,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,617,528,23706,325776,0.072768,1,1,train,42


Unique images: 1000
Mean mask area ratio: 0.15390989411452588
